In [ ]:
import matplotlib
matplotlib.use("Agg")
import torch
from torch import nn
from matplotlib import pyplot as plt
import gc
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import getpass
import random
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir, "Datasets","Car Prices.csv")
df = pd.read_csv(df_path)
df = df.dropna()

x = df.drop(columns="car_price").values
y = df["car_price"].values.reshape(-1,1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.23, random_state=12)

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

user_account = getpass.getuser()
predictions_bar_chart = os.path.join(script_dir, f"Car Price Predictor Bar Chart.png")

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")


class Regressor(nn.Module):
    def __init__(self, input_dim=x_train.shape[1]):
        super().__init__()
        self.network = nn.Sequential (
            nn.Linear(in_features=input_dim, out_features=18),
            nn.LeakyReLU(),
            nn.BatchNorm1d(18),

            nn.Linear(in_features=18, out_features=31),
            nn.LeakyReLU(),
            nn.BatchNorm1d(31),

            nn.Linear(in_features=31, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),

            nn.Linear(in_features=42, out_features=53),
            nn.LeakyReLU(),
            nn.BatchNorm1d(53),
            nn.Dropout(p=0.5),

            nn.Linear(in_features=53, out_features=67),
            nn.LeakyReLU(),
            nn.BatchNorm1d(67),
            nn.Dropout(p=0.8),

            nn.Linear(in_features=67, out_features=78),
            nn.LeakyReLU(),
            nn.BatchNorm1d(78),
            nn.Dropout(p=0.2),

            nn.Linear(in_features=78, out_features=58),
            nn.LeakyReLU(),
            nn.BatchNorm1d(58),

            nn.Linear(in_features=58, out_features=32),
            nn.LeakyReLU(),
            nn.BatchNorm1d(32),

            nn.Linear(in_features=32, out_features=1),

        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)  

losses = []
def run_grid_search(epochs_list, rates_list, seed_val):
    
    best_State = None
    loss_function = nn.L1Loss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15}")
    print(f"=" * 65)

    for lr in rates_list:
        for epoch_count in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = Regressor().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_function(y_preds, y_train)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_loss = loss_function(model(x_test), y_test)
            print(f"{lr:<15} | {epoch_count:<15} | {loss:<15}")
            losses.append(test_loss)
            import copy
            best_State = copy.deepcopy(model.state_dict())

            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.3f}")
    print(f"● Worst Loss = {max(losses):.3f}")
    return best_State               
 
epoch_counts = [340,670,450,980]
rates = [0.0001, 0.001, 0.01, 0.1]

seed_val = float(input("-- Enter a Value for a Seed : ").strip())
best_state = run_grid_search(epoch_counts, rates, seed_val)
torch.save(best_state, f"Car Price Predictor Parameters.pth")

fresh_model = Regressor().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip().title()
    if user_name.lower() == "exit":
        print(f"Exiting Interactive Loop")
        break
    engine_cc = float(input("-- Enter your Engine CC : ").strip())
    mileage_km = float(input("-- Enter your Preferred milage in Kilometers : ").strip())
    age_years = float(input("-- How old do you want your car to be : ").strip())
    horsepower = float(input("-- Enter your Preferred Horse Power : ").strip())
    weight_kg = float(input("-- How much do you want your Car to weigh in Kilograms : ").strip())

    raw_array = np.array([engine_cc, mileage_km, age_years, horsepower, weight_kg])
    scaled_array = x_scaler.fit_transform(raw_array)
    scaled_tensor = torch.tensor(scaled_array, dtype=torch.float32).to(device)

    with torch.inference_mode():
        preds = fresh_model(scaled_tensor).cpu().numpy()
    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted Car Price for you, {user_name} = {round(untransformed)} $")
    session_inputs.append(f"{user_name} ( {mileage_km} Km )")
    session_predictions.append(round(untransformed))

try:
    if session_predictions and session_inputs:
        plt.figure(figsize=(15,4))
        colors = ["red","blue","green","orange","purple"]
        bar_colors = [random.choice(colors) for _ in range(len(session_predictions))]
        plt.bar(session_inputs, session_predictions, color=bar_colors, edgecolor="black")
        plt.title("Predictions Bar Chart = Car Price Predictor")
        plt.xlabel("Person | Mileage")
        plt.ylabel("Predicted Car Prices")
        plt.grid()
        plt.legend(prop={"size":14,"weight":"bold"})
        plt.savefig(predictions_bar_chart)
        plt.close()
        print(f"Saved the Graph")
    else:
        print(f"Failed to Save Graph = No Inputs or Predictions")
except BaseException as e:
    print(f"Failed to Save Graph = {e}") 

one_drive = input("-- Do you use One Drive or Not ( Yes / No ) : ").strip().title()
try:
    if one_drive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Car Price Predictor Details.txt"      
    elif one_drive == "No":
        desktop_path =  f"C:/Users/{user_account}/Desktop/Car Price Predictor Details.txt" 
except BaseException as e:
    print(f"Please enter Valid Input. Error has occured = {e}")

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
        file.write(f"== Car Price Predictor Details ==\n")
        file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")
        file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}")
        file.write(f"● Worst Loss = {min(losses):.2f}\n")
        file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"Sucessfully Saved your Model Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")                   